## 04 — Race Strategy and Pit Stop Modeling

This notebook extends the project from pre-race forecasting into in-race tactical modeling.

The goal is to evaluate how pit stop execution, lap-time behavior, sprint context, and weather interact with pre-race expectations to influence final performance.

At the center of this notebook is one core question:

**How much can in-race strategy shift outcomes relative to pre-race expectations?**

This notebook focuses on four tactical targets:

- **finish_improvement**: finishing position improvement relative to grid
- **outperform_quali**: whether a driver finished ahead of qualifying expectation
- **finish_residual**: deviation from model-based expected finish
- **overperformed_model**: whether a driver beat the model-based expected finish

Because the notebook evaluates multiple related targets, it is designed as a reusable target-driven modeling framework.

This allows the same core workflow — data preparation, model fitting, evaluation, interpretability, and visualization — to be applied consistently across several tactical outcome definitions.

### Targets used in this notebook

**Finish improvement relative to grid**

- `finish_improvement = grid_clean - finish_position`

**Outperform qualifying expectation**

- `outperform_quali = (finish_position < qualifying_position).astype(int)`

**Finish vs Expected Finish (Residual-Based Target)**

- `finish_residual = expected_finish - finish_position`

**Overperformance vs Expected (Binary)**

- `overperformed_model = (finish_position < expected_finish).astype(int)`

### Future extensions

Later iterations can expand this framework to additional strategy-oriented targets such as:
- pit efficiency
- pace relative to field
- lap-time consistency
- strategy gain vs expected gain

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor, XGBClassifier

import shap

In [2]:
strategy = pd.read_parquet("data_processed/f1_feature_store_strategy.parquet")

strategy.shape
strategy.head()

,raceId,driverId,constructorId,circuitId,date,year,round,target_points,finish_position,positionOrder,...,avg_lap_time_ms,median_lap_time_ms,std_lap_time_ms,fastest_lap_in_race_ms,slowest_lap_ms,milliseconds,laps,fastestLap,fastestLapTime,fastestLapSpeed
0,18,1,1,1,2008-03-16,2008,1,1,1,1,...,98114.068966,88665.5,20990.356845,87452.0,166432.0,5690616.0,58,39,1:27.452,218.300
1,18,2,2,1,2008-03-16,2008,1,1,2,2,...,98208.517241,89388.0,20212.298423,87739.0,166678.0,5696094.0,58,41,1:27.739,217.586
2,18,3,3,1,2008-03-16,2008,1,1,3,3,...,98254.810345,89585.0,18957.199698,88090.0,156683.0,5698779.0,58,41,1:28.090,216.719
3,18,4,4,1,2008-03-16,2008,1,1,4,4,...,98410.293103,90668.5,19640.823974,88603.0,170306.0,5707797.0,58,58,1:28.603,215.464
4,18,5,1,1,2008-03-16,2008,1,1,5,5,...,98424.655172,89442.5,21826.401776,87418.0,175160.0,5708630.0,58,43,1:27.418,218.385


In [3]:
strategy.columns.tolist()

['raceId',
 'driverId',
 'constructorId',
 'circuitId',
 'date',
 'year',
 'round',
 'target_points',
 'finish_position',
 'positionOrder',
 'points',
 'grid',
 'grid_clean',
 'qualifying_position',
 'best_qualifying_time_ms',
 'made_q2',
 'made_q3',
 'driver_avg_finish_last5',
 'driver_points_last5',
 'driver_dnf_rate_last5',
 'driver_avg_grid_last5',
 'constructor_points_last5',
 'constructor_dnf_rate_last5',
 'constructor_avg_finish_last5',
 'driver_standing_points_prerace',
 'driver_standing_position_prerace',
 'driver_standing_wins_prerace',
 'constructor_standing_points_prerace',
 'constructor_standing_position_prerace',
 'constructor_standing_wins_prerace',
 'lat',
 'lng',
 'alt',
 'abs_lat',
 'temp_avg',
 'temp_max',
 'temp_min',
 'temp_range',
 'precipitation',
 'wind_speed',
 'humidity_avg',
 'month',
 'is_wet_race',
 'high_altitude_track',
 'sprint_grid',
 'sprint_position_order',
 'sprint_points',
 'sprint_laps',
 'sprint_milliseconds',
 'pit_stop_count',
 'avg_pit_stop_ms'

In [4]:
FEATURE_GROUPS = {
    "tactical_core": [
        "pit_stop_count",
        "avg_pit_stop_ms",
        "total_pit_stop_ms",
        "first_pit_lap",
        "last_pit_lap",
        "avg_lap_time_ms",
        "median_lap_time_ms",
        "std_lap_time_ms",
        "fastest_lap_in_race_ms",
        "lap_count_recorded"
    ],
    "contextual": [
        "alt",
        "high_altitude_track",
        "year"
    ],
    "pre_race_controls": [
        "grid_clean",
        "qualifying_position",
        "driver_avg_finish_last5",
        "constructor_points_last5"
    ],
    "weather": [
        "temp_avg",
        "precipitation",
        "wind_speed",
        "is_wet_race"
    ],
    "sprint": [
        "sprint_position_order",
        "sprint_points"
    ]
}

In [5]:
features_04 = (
    FEATURE_GROUPS["tactical_core"]
    + FEATURE_GROUPS["contextual"]
    + FEATURE_GROUPS["pre_race_controls"]
    + FEATURE_GROUPS["weather"]
    + FEATURE_GROUPS["sprint"]
)

In [6]:
missing_cols = [c for c in features_04 if c not in strategy.columns]
missing_cols

[]

In [7]:
df = strategy.copy()

df["finish_improvement"] = df["grid_clean"] - df["finish_position"]
df["outperform_quali"] = (df["finish_position"] < df["qualifying_position"]).astype(int)

In [8]:
[c for c in ["expected_finish", "finish_position", "grid_clean", "qualifying_position"] if c in df.columns]

['finish_position', 'grid_clean', 'qualifying_position']

## Generate Expected Finish (Pre-Race Baseline)

To evaluate how in-race strategy shifts outcomes, we first need a baseline expectation of performance.

Because `expected_finish` is not stored in the feature store, we recreate it here using a pre-race regression model (consistent with Notebook 03).

This ensures:
- consistency with prior modeling assumptions
- no data leakage
- a self-contained modeling pipeline

The predicted values from this model serve as the baseline expectation against which tactical performance is evaluated.

In [9]:
pre_race_features = [
    "grid_clean",
    "qualifying_position",
    "driver_avg_finish_last5",
    "driver_points_last5",
    "driver_dnf_rate_last5",
    "constructor_points_last5",
    "constructor_dnf_rate_last5",
    "constructor_avg_finish_last5",
    "driver_standing_points_prerace",
    "driver_standing_position_prerace",
    "constructor_standing_points_prerace",
    "constructor_standing_position_prerace",
    "year"
]

In [10]:
[c for c in pre_race_features if c not in df.columns]

[]

In [11]:
#build dataset
baseline_df = df[["finish_position"] + pre_race_features].copy()

for col in baseline_df.columns:
    baseline_df[col] = pd.to_numeric(baseline_df[col], errors="coerce")

baseline_df = baseline_df.dropna().copy()
baseline_df.shape

(9567, 14)

In [12]:
#train/test split
train_base = baseline_df[baseline_df["year"] <= 2021].copy()
test_base = baseline_df[baseline_df["year"] > 2021].copy()

X_train_base = train_base[pre_race_features]
y_train_base = train_base["finish_position"]

X_test_base = test_base[pre_race_features]
y_test_base = test_base["finish_position"]

In [13]:
#train model
baseline_model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

baseline_model.fit(X_train_base, y_train_base)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [14]:
#generate expected finish for all rows in full dataset
X_full_base = baseline_df[pre_race_features]

baseline_model.fit(X_full_base, baseline_df["finish_position"])
baseline_df["expected_finish"] = baseline_model.predict(X_full_base)

In [15]:
#merge back to main dataset
df = df.merge(
    baseline_df[["expected_finish"]],
    left_index=True,
    right_index=True,
    how="left"
)

In [16]:
df["finish_residual"] = df["expected_finish"] - df["finish_position"]

df["overperformed_model"] = (
    df["finish_position"] < df["expected_finish"]
).astype(int)

In [17]:
df[["expected_finish", "finish_residual"]].describe()

,expected_finish,finish_residual
count,9567.000000,9567.000000
mean,10.823792,0.004203
std,3.978334,4.453529
min,0.889684,-18.021708
25%,7.807323,-2.290812
50%,11.217068,0.980543
75%,13.834049,3.030578
max,21.174866,11.576746


## Finalize Tactical Targets

With `expected_finish` reconstructed, we can now define residual-based and model-relative tactical targets.

In [18]:
df["finish_residual"] = df["expected_finish"] - df["finish_position"]
df["overperformed_model"] = (df["finish_position"] < df["expected_finish"]).astype(int)

df[[
    "finish_improvement",
    "outperform_quali",
    "finish_residual",
    "overperformed_model"
]].describe()

,finish_improvement,outperform_quali,finish_residual,overperformed_model
count,25121.000000,26759.000000,9567.000000,26759.000000
mean,0.002189,0.200306,0.004203,0.214582
std,7.185223,0.400237,4.453529,0.410540
min,-31.000000,0.000000,-18.021708,0.000000
25%,-4.000000,0.000000,-2.290812,0.000000
50%,1.000000,0.000000,0.980543,0.000000
75%,4.000000,0.000000,3.030578,0.000000
max,30.000000,1.000000,11.576746,1.000000


In [19]:
#binary rate
target_balance_04 = pd.DataFrame({
    "target": ["outperform_quali", "overperformed_model"],
    "positive_rate": [
        df["outperform_quali"].mean(),
        df["overperformed_model"].mean()
    ]
})

target_balance_04

,target,positive_rate
0,outperform_quali,0.200306
1,overperformed_model,0.214582


## Part 1 — Setup and target registry

Define a dictionary that stores metadata for each target.

## Target Registry

Define a reusable registry describing each tactical modeling target, including problem type, label, and evaluation metrics.

In [20]:
TARGET_SPECS = {
    "finish_improvement": {
        "type": "regression",
        "label": "Finish Improvement Relative to Grid",
        "target_col": "finish_improvement",
        "baseline_model": "linear_regression",
        "tree_model": "xgboost_regressor",
        "metrics": ["rmse", "mae", "r2"]
    },
    "outperform_quali": {
        "type": "classification",
        "label": "Outperform Qualifying Expectation",
        "target_col": "outperform_quali",
        "baseline_model": "logistic_regression",
        "tree_model": "xgboost_classifier",
        "metrics": ["roc_auc", "calibration", "lift"]
    },
    "finish_residual": {
        "type": "regression",
        "label": "Finish vs Expected Finish Residual",
        "target_col": "finish_residual",
        "baseline_model": "linear_regression",
        "tree_model": "xgboost_regressor",
        "metrics": ["rmse", "mae", "r2"]
    },
    "overperformed_model": {
        "type": "classification",
        "label": "Overperformed Model Expectation",
        "target_col": "overperformed_model",
        "baseline_model": "logistic_regression",
        "tree_model": "xgboost_classifier",
        "metrics": ["roc_auc", "calibration", "lift"]
    }
}

## Part 2 — Shared preprocessing function

function to:

- select columns
- coerce numeric types
- drop missing rows
- perform temporal split

## Shared Data Preparation Function

Build a reusable preparation function that applies numeric conversion, missing-value handling, and temporal train/test splitting consistently across targets.

In [29]:
def prepare_model_data(df, target_col, feature_cols, split_year=2021):
    keep_cols = ["raceId", "driverId", "year", target_col] + feature_cols
    model_df = df[keep_cols].copy()

    for col in model_df.columns:
        if col not in ["raceId", "driverId"]:
            model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

    model_df = model_df.dropna().copy()

    train = model_df[model_df["year"] <= split_year].copy()
    test = model_df[model_df["year"] > split_year].copy()

    X_train = train[feature_cols]
    y_train = train[target_col]
    X_test = test[feature_cols]
    y_test = test[target_col]

    return {
        "model_df": model_df,
        "train": train,
        "test": test,
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test
    }

## Model Functions

## Shared Model Runners

Create reusable modeling functions for regression and classification targets so the same framework can be applied consistently across tactical outcome definitions.

In [41]:
from sklearn.calibration import calibration_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [42]:
#Regression
def run_regression_models(X_train, y_train, X_test, y_test, random_state=42):
    results = {}

    linreg = Pipeline([
        ("scaler", StandardScaler()),
        ("linreg", LinearRegression())
    ])
    linreg.fit(X_train, y_train)
    preds_lin = linreg.predict(X_test)

    results["linear_regression"] = {
        "model": linreg,
        "preds": preds_lin,
        "rmse": np.sqrt(mean_squared_error(y_test, preds_lin)),
        "mae": mean_absolute_error(y_test, preds_lin),
        "r2": r2_score(y_test, preds_lin),
        "residuals": y_test - preds_lin
    }

    xgb_reg = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=random_state
    )
    xgb_reg.fit(X_train, y_train)
    preds_xgb = xgb_reg.predict(X_test)

    results["xgboost"] = {
        "model": xgb_reg,
        "preds": preds_xgb,
        "rmse": np.sqrt(mean_squared_error(y_test, preds_xgb)),
        "mae": mean_absolute_error(y_test, preds_xgb),
        "r2": r2_score(y_test, preds_xgb)
    }

    return results

In [32]:
#Classification
def run_classification_models(X_train, y_train, X_test, y_test, random_state=42):
    results = {}

    logit = Pipeline([
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(max_iter=2000))
    ])
    logit.fit(X_train, y_train)
    probs_logit = logit.predict_proba(X_test)[:, 1]

    prob_true, prob_pred = calibration_curve(y_test, probs_logit, n_bins=10)

    results["logistic_regression"] = {
        "model": logit,
        "probs": probs_logit,
        "roc_auc": roc_auc_score(y_test, probs_logit),
        "calibration_true": prob_true,
        "calibration_pred": prob_pred
    }

    xgb_cls = XGBClassifier(
        n_estimators=250,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=random_state,
        eval_metric="logloss"
    )
    xgb_cls.fit(X_train, y_train)
    probs_xgb = xgb_cls.predict_proba(X_test)[:, 1]

    results["xgboost"] = {
        "model": xgb_cls,
        "probs": probs_xgb,
        "roc_auc": roc_auc_score(y_test, probs_xgb)
    }

    return results

## Part 3 — Separate modeling functions by problem type
For regression

Create a shared function that:

- fits linear baseline
- fits XGBoost regressor
- returns metrics
- returns residuals
- returns feature importance / SHAP
- For classification

Create a shared function that:

- fits logistic regression baseline
- fits XGBoost classifier
- returns ROC-AUC
- returns calibration data
- returns lift data
- returns SHAP

## Part 4 — Target loop
**Run Target-Driven Modeling Loop**

Iterate through the target registry and apply the shared preparation and modeling workflow to each target.

In [43]:
def prepare_model_data(df, target_col, feature_cols, split_year=2021):
    # remove duplicates while preserving order
    keep_cols = ["raceId", "driverId", "year", target_col] + feature_cols
    keep_cols = list(dict.fromkeys(keep_cols))

    model_df = df[keep_cols].copy()

    for col in model_df.columns:
        if col not in ["raceId", "driverId"]:
            model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

    model_df = model_df.dropna().copy()

    train = model_df[model_df["year"] <= split_year].copy()
    test = model_df[model_df["year"] > split_year].copy()

    X_train = train[feature_cols]
    y_train = train[target_col]
    X_test = test[feature_cols]
    y_test = test[target_col]

    return {
        "model_df": model_df,
        "train": train,
        "test": test,
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test
    }

In [44]:
all_results = {}

feature_cols_04 = [c for c in features_04 if c in df.columns]

for target_name, spec in TARGET_SPECS.items():
    print(f"Running target: {target_name}")

    prepared = prepare_model_data(
        df=df,
        target_col=spec["target_col"],
        feature_cols=feature_cols_04,
        split_year=2021
    )

    if spec["type"] == "regression":
        model_results = run_regression_models(
            prepared["X_train"],
            prepared["y_train"],
            prepared["X_test"],
            prepared["y_test"]
        )
    else:
        model_results = run_classification_models(
            prepared["X_train"],
            prepared["y_train"],
            prepared["X_test"],
            prepared["y_test"]
        )

    all_results[target_name] = {
        "spec": spec,
        "prepared": prepared,
        "results": model_results
    }

Running target: finish_improvement
Running target: outperform_quali
Running target: finish_residual
Running target: overperformed_model


In [37]:
keep_cols = ["raceId", "driverId", "year", "finish_improvement"] + feature_cols_04
dupes = pd.Series(keep_cols).value_counts()
dupes[dupes > 1]

year    2
Name: count, dtype: int64

## Summarize Model Performance Across Targets

Create unified summary tables for regression and classification targets.

In [45]:
#Regression
regression_rows = []

for target_name, payload in all_results.items():
    if payload["spec"]["type"] == "regression":
        for model_name, res in payload["results"].items():
            regression_rows.append({
                "target": target_name,
                "model": model_name,
                "rmse": res["rmse"],
                "mae": res["mae"],
                "r2": res["r2"]
            })

regression_summary = pd.DataFrame(regression_rows)
regression_summary

,target,model,rmse,mae,r2
0,finish_improvement,linear_regression,396.281053,314.538461,-6949.656526
1,finish_improvement,xgboost,4.827843,3.608954,-0.031633
2,finish_residual,linear_regression,646.004789,512.106651,-30901.173790
3,finish_residual,xgboost,4.292978,3.266260,-0.364693


In [39]:
#Classification
classification_rows = []

for target_name, payload in all_results.items():
    if payload["spec"]["type"] == "classification":
        for model_name, res in payload["results"].items():
            classification_rows.append({
                "target": target_name,
                "model": model_name,
                "roc_auc": res["roc_auc"]
            })

classification_summary = pd.DataFrame(classification_rows)
classification_summary

,target,model,roc_auc
0,outperform_quali,logistic_regression,0.646957
1,outperform_quali,xgboost,0.636812
2,overperformed_model,logistic_regression,0.616517
3,overperformed_model,xgboost,0.594623
